In [10]:
import os
import sys
import json

In [11]:
DATA_PATH = '../' 'nogit/sequence_noise/rzrxrz/1q_100g_100blk_data/'
GOOD_DATA_PATH = DATA_PATH + 'good_fidelity/'

good_data = []
for filename in os.listdir(GOOD_DATA_PATH):
    if not filename.endswith('.json'):
        continue

    filepath = os.path.join(GOOD_DATA_PATH, filename)
    with open(filepath, 'r') as f:
        data = json.load(f)
        good_data.append(data)

print(f'Loaded {len(good_data)} good fidelity data files.', file=sys.stderr)

Loaded 1001 good fidelity data files.


In [12]:
print(good_data[0].keys())

print(good_data[0]['pqc_params'])

print(good_data[0]['base_circuit_tokens'])
print(good_data[0]['pqc_circuit_tokens'])

dict_keys(['seed', 'fidelity', 'pqc_params', 'base_circuit_tokens', 'pqc_circuit_tokens', 'base_circuit_qasm', 'pqc_circuit_qasm'])
{'pre_angles': [[[-1.5675233602523804, 3.141535520553589, 1.5740692615509033]]]}
[['z', [0], []], ['x', [0], []], ['x', [0], []], ['z', [0], []], ['z', [0], []], ['h', [0], []], ['h', [0], []], ['x', [0], []], ['z', [0], []], ['h', [0], []], ['h', [0], []], ['h', [0], []], ['x', [0], []], ['h', [0], []], ['z', [0], []], ['z', [0], []], ['x', [0], []], ['z', [0], []], ['x', [0], []], ['z', [0], []], ['h', [0], []], ['z', [0], []], ['h', [0], []], ['h', [0], []], ['z', [0], []], ['z', [0], []], ['z', [0], []], ['x', [0], []], ['x', [0], []], ['h', [0], []], ['h', [0], []], ['z', [0], []], ['x', [0], []], ['z', [0], []], ['z', [0], []], ['h', [0], []], ['x', [0], []], ['z', [0], []], ['h', [0], []], ['h', [0], []], ['x', [0], []], ['h', [0], []], ['z', [0], []], ['x', [0], []], ['h', [0], []], ['h', [0], []], ['x', [0], []], ['h', [0], []], ['z', [0], []], ['

In [13]:
filtered_data = []
same_data_different_angles = []

In [14]:
import numpy as np

def make_hashable(obj):
    """Recursively convert lists to tuples to make objects hashable."""
    if isinstance(obj, list):
        return tuple(make_hashable(item) for item in obj)
    elif isinstance(obj, dict):
        return tuple(sorted((k, make_hashable(v)) for k, v in obj.items()))
    else:
        return obj

# Dictionary to track circuits by their base_circuit_tokens
circuit_map = {}

for data in good_data:
    # Convert base_circuit_tokens to a hashable key
    base_tokens = make_hashable(data['base_circuit_tokens'])
    
    if base_tokens not in circuit_map:
        # First occurrence of this circuit
        circuit_map[base_tokens] = data
    else:
        # Circuit already exists, check if PQC params differ
        existing_data = circuit_map[base_tokens]
        
        # Extract PQC parameters for comparison
        existing_params = existing_data['pqc_params']
        current_params = data['pqc_params']
        
        # Handle different parameter structures (dict or array)
        if isinstance(existing_params, dict):
            # If it's a dict with 'pre_angles' or similar keys
            if 'pre_angles' in existing_params:
                existing_angles = np.array(existing_params['pre_angles']).flatten()
                current_angles = np.array(current_params['pre_angles']).flatten()
            else:
                # Convert all dict values to flat arrays
                existing_angles = np.concatenate([np.array(v).flatten() for v in existing_params.values()])
                current_angles = np.concatenate([np.array(v).flatten() for v in current_params.values()])
        else:
            # Assume it's already array-like
            existing_angles = np.array(existing_params).flatten()
            current_angles = np.array(current_params).flatten()
        
        # Check if parameters differ by more than threshold
        max_diff = np.max(np.abs(existing_angles - current_angles))
        
        if max_diff > 0.001:
            # Record this pair as having different angles
            same_data_different_angles.append((existing_data, data))
            print(f"Found duplicate circuit with different angles: max_diff = {max_diff:.6f}")

# Extract filtered data (unique circuits only)
filtered_data = list(circuit_map.values())

print("\nFiltering Summary:")
print(f"  Original data: {len(good_data)} circuits")
print(f"  Unique circuits: {len(filtered_data)} circuits")
print(f"  Duplicate circuits with different angles: {len(same_data_different_angles)} pairs")


Filtering Summary:
  Original data: 1001 circuits
  Unique circuits: 1001 circuits
  Duplicate circuits with different angles: 0 pairs


In [15]:
# Convert PQC params from dict to list in filtered_data
for data in filtered_data:
    if isinstance(data['pqc_params'], dict):
        # Extract the angles and flatten to a list
        if 'pre_angles' in data['pqc_params']:
            # Convert pre_angles to a flat list
            data['pqc_params'] = np.array(data['pqc_params']['pre_angles']).flatten().tolist()
        else:
            # Concatenate all dict values into a single list
            all_params = []
            for key in sorted(data['pqc_params'].keys()):  # Sort for consistency
                all_params.extend(np.array(data['pqc_params'][key]).flatten().tolist())
            data['pqc_params'] = all_params

print(f"Converted PQC params to list format for {len(filtered_data)} circuits")
print("\nExample PQC params (now as list):")
print(f"  Type: {type(filtered_data[0]['pqc_params'])}")
print(f"  Length: {len(filtered_data[0]['pqc_params'])}")
print(f"  First few values: {filtered_data[0]['pqc_params'][:5]}")

Converted PQC params to list format for 1001 circuits

Example PQC params (now as list):
  Type: <class 'list'>
  Length: 3
  First few values: [-1.5675233602523804, 3.141535520553589, 1.5740692615509033]


In [16]:
UNIQUE_DATA_FILE_PATH = DATA_PATH + f'unique_good_fidelity_{DATA_PATH.split("/")[-2]}.json'
with open(UNIQUE_DATA_FILE_PATH, 'w') as f:
    json.dump(sorted(filtered_data, key=lambda x: x['seed']), f)

In [17]:
# Examine some examples of circuits with different angles
if len(same_data_different_angles) > 0:
    print("\nExample of duplicate circuit with different PQC parameters:")
    print("="*60)
    pair = same_data_different_angles[0]

    print("\nBase circuit tokens (same for both):")
    print(pair[0]['base_circuit_tokens'][:5], "...")  # Show first 5 tokens

    print("\nFirst instance PQC params:")
    print(pair[0]['pqc_params'])

    print("\nSecond instance PQC params:")
    print(pair[1]['pqc_params'])
    
    # Calculate difference
    if isinstance(pair[0]['pqc_params'], dict) and 'pre_angles' in pair[0]['pqc_params']:
        angles1 = np.array(pair[0]['pqc_params']['pre_angles']).flatten()
        angles2 = np.array(pair[1]['pqc_params']['pre_angles']).flatten()
        print("\nAngle differences:")
        print(angles2 - angles1)
else:
    print("\nNo duplicate circuits with different angles found.")


No duplicate circuits with different angles found.


In [18]:
# Inspect a JSON file properly
# Replace with your actual file path
json_file_path = UNIQUE_DATA_FILE_PATH  # or use a specific path

with open(json_file_path, 'r') as f:
    data = json.load(f)

# Pretty print the first item if it's a list, or the whole thing if it's a dict
if isinstance(data, list) and len(data) > 0:
    print(json.dumps(data[0], indent=2))
else:
    print(json.dumps(data, indent=2))

{
  "seed": 0,
  "fidelity": 1.0,
  "pqc_params": [
    1.8358230590820312e-05,
    0.0,
    0.0
  ],
  "base_circuit_tokens": [
    [
      "z",
      [
        0
      ],
      []
    ],
    [
      "z",
      [
        0
      ],
      []
    ],
    [
      "h",
      [
        0
      ],
      []
    ],
    [
      "x",
      [
        0
      ],
      []
    ],
    [
      "h",
      [
        0
      ],
      []
    ],
    [
      "h",
      [
        0
      ],
      []
    ],
    [
      "z",
      [
        0
      ],
      []
    ],
    [
      "x",
      [
        0
      ],
      []
    ],
    [
      "h",
      [
        0
      ],
      []
    ],
    [
      "h",
      [
        0
      ],
      []
    ],
    [
      "z",
      [
        0
      ],
      []
    ],
    [
      "h",
      [
        0
      ],
      []
    ],
    [
      "x",
      [
        0
      ],
      []
    ],
    [
      "z",
      [
        0
      ],
      []
    ],
    [
      "h",
      [
      